# 0) Imports & API Key Setup
Load OpenAI API key the same way we do in accuracy testing.

In [8]:
import json, time
from pathlib import Path

# Find repo root
def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start).resolve()
    for p in [start] + list(start.parents):
        if (p / ".git").exists() or (p / "README.md").exists():
            return p
    raise RuntimeError(f"Repo root not found from start={start}")

repo_root = find_repo_root()
print("Repo root:", repo_root)

# Load API key
KEYS_PATH = repo_root / "keys.json"
if not KEYS_PATH.exists():
    raise FileNotFoundError(f"keys.json missing: {KEYS_PATH}")

keys = json.loads(KEYS_PATH.read_text())
OPENAI_API_KEY = keys.get("openai")
if not OPENAI_API_KEY:
    raise ValueError("Missing 'openai' key in keys.json")

# Install + init OpenAI
%pip install -q openai
from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY)

print("OpenAI client initialized.")

Repo root: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/ici01_PublicOpinionAnalyzer

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
OpenAI client initialized.


# 1) Paths & Split & Model Selection
Here you choose the split and provider/model settings.

In [9]:
from pathlib import Path
from datetime import datetime, timezone
import re

# ------------------------------------------------------------------
# Dataset root
# ------------------------------------------------------------------
dataset_root = repo_root / "datasets" / "yt_factlink"
print("Dataset root:", dataset_root)

# ------------------------------------------------------------------
# Choose split
# ------------------------------------------------------------------
SPLIT_NAME = "split_v1_seed42"   # <--- Change here to switch splits

# ------------------------------------------------------------------
# Choose provider + base model
# ------------------------------------------------------------------
PROVIDER    = "openai"
MODEL_NAME  = "gpt-4.1-2025-04-14"   # model to fine-tune FROM
SUFFIX_TAG  = "all_at_once"          # your experiment name

# This becomes the suffix in the OpenAI dashboard
FINETUNE_SUFFIX = f"{SUFFIX_TAG}_{SPLIT_NAME}"

# ------------------------------------------------------------------
# Split paths
# ------------------------------------------------------------------
SPLIT_DIR   = dataset_root / "01_conversion" / "03_splits" / SPLIT_NAME
TRAIN_JSONL = SPLIT_DIR / "train.data.jsonl"
TEST_JSONL  = SPLIT_DIR / "test.data.jsonl"

assert SPLIT_DIR.exists(),   f"Missing split folder: {SPLIT_DIR}"
assert TRAIN_JSONL.exists(), f"Missing train split: {TRAIN_JSONL}"
assert TEST_JSONL.exists(),  f"Missing test split: {TEST_JSONL}"

print("Using split:", SPLIT_NAME)
print("Train split:", TRAIN_JSONL)
print("Test  split:", TEST_JSONL)

# ------------------------------------------------------------------
# Prompt paths
# ------------------------------------------------------------------
PROMPT_DIR  = dataset_root / "02_prompts" / SUFFIX_TAG
SYSTEM_PATH = PROMPT_DIR / "system.txt"
USER_PATH   = PROMPT_DIR / "user.txt"

assert SYSTEM_PATH.exists(), f"Missing system prompt: {SYSTEM_PATH}"
assert USER_PATH.exists(),   f"Missing user prompt: {USER_PATH}"

print("Prompts loaded from:", PROMPT_DIR)

# ------------------------------------------------------------------
# Create global "runs" folder next to this notebook
# Then create ONE folder for this specific run inside it.
# ------------------------------------------------------------------

import re

def slugify(s: str) -> str:
    s = re.sub(r"[^a-zA-Z0-9._-]+", "-", s)
    s = re.sub(r"-{2,}", "-", s)
    return s.strip("-")

# Folder where the notebook lives
NOTEBOOK_DIR = Path.cwd()

# Create folder for this model (next to this notebook)
MODEL_DIR = NOTEBOOK_DIR / slugify(MODEL_NAME)
MODEL_DIR.mkdir(exist_ok=True)

# Add .gitkeep at the model folder level
(MODEL_DIR / ".gitkeep").write_text("")

# Create runs/ folder inside model folder
RUNS_ROOT = MODEL_DIR / "runs"
RUNS_ROOT.mkdir(exist_ok=True)

# Add .gitkeep inside runs folder
(RUNS_ROOT / ".gitkeep").write_text("")

# Timestamp-based unique run name
timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H%M%S")

# Smart run folder name
run_name = "_".join([
    timestamp,
    slugify(PROVIDER),
    slugify(MODEL_NAME),
    slugify(SUFFIX_TAG),
    slugify(SPLIT_NAME),
])

# 5) Create the run folder inside runs/
RUN_DIR = RUNS_ROOT / run_name
RUN_DIR.mkdir(parents=True, exist_ok=False)


print("Notebook directory:", NOTEBOOK_DIR.resolve())
print("Global runs folder:", RUNS_ROOT.resolve())
print("Run directory     :", RUN_DIR.resolve())

# ------------------------------------------------------------------
# Derived training file — stored INSIDE the run folder
# ------------------------------------------------------------------
OPENAI_TRAIN_FILE = RUN_DIR / "train.openai.jsonl"
print("Derived training file will be:", OPENAI_TRAIN_FILE)

# Summary
print("\n=== Configuration OK ===")
print("Base model         :", MODEL_NAME)
print("Provider           :", PROVIDER)
print("Finetune Suffix    :", FINETUNE_SUFFIX)
print("Run directory      :", RUN_DIR)


Dataset root: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/ici01_PublicOpinionAnalyzer/datasets/yt_factlink
Using split: split_v1_seed42
Train split: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/01_conversion/03_splits/split_v1_seed42/train.data.jsonl
Test  split: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/01_conversion/03_splits/split_v1_seed42/test.data.jsonl
Prompts loaded from: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/02_prompts/all_at_once
Notebook directory: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/03_finetuning/all_at_once/ope

# 2) Load Split + Prompts
Load train split and the frozen finetuning prompt pair.

In [12]:
def read_jsonl(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train_rows = read_jsonl(TRAIN_JSONL)
print("Train examples:", len(train_rows))

system_template = SYSTEM_PATH.read_text(encoding="utf-8")
user_template   = USER_PATH.read_text(encoding="utf-8")

# ---------------------------------------------------------------
# Copy prompt files into RUN_DIR/01_inputs for reproducibility
# ---------------------------------------------------------------
INPUTS_DIR = RUN_DIR / "01_inputs"
INPUTS_DIR.mkdir(parents=True, exist_ok=True)

# Write frozen copies
(INPUTS_DIR / "system.txt").write_text(system_template, encoding="utf-8")
(INPUTS_DIR / "user.txt").write_text(user_template, encoding="utf-8")

print("Copied prompts to:", INPUTS_DIR.resolve())

Train examples: 309
Copied prompts to: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/03_finetuning/all_at_once/openai/gpt-4.1-2025-04-14/runs/2025-11-28_045222_openai_gpt-4.1-2025-04-14_all_at_once_split_v1_seed42/01_inputs


# 3) Build OpenAI Fine-Tuning JSONL
Convert each canonical example into messages format.

In [13]:
# Add .gitkeep inside individual run folder
(RUN_DIR / ".gitkeep").write_text("")

# Create subfolder for this run's inputs
INPUTS_DIR = RUN_DIR / "01_inputs"
INPUTS_DIR.mkdir(parents=True, exist_ok=True)

# Path for the training file used for fine-tuning
OPENAI_TRAIN_FILE = INPUTS_DIR / "train.openai.jsonl"

print("Input directory created at:", INPUTS_DIR.resolve())
print("Training file will be:", OPENAI_TRAIN_FILE.resolve())

# ---------------------------------------------------------
# Build the OpenAI training JSONL
# ---------------------------------------------------------

def build_user_message(r):
    msg = user_template
    msg = msg.replace("[VIDEOTITLE]",   r.get("VIDEOTITEL", ""))
    msg = msg.replace("[VIDEOCONTEXT]", r.get("VIDEOCONTEXT", ""))
    msg = msg.replace("[TARGETCOMMENT]", r.get("TARGETCOMMENT", ""))
    return msg

openai_lines = []
for r in train_rows:
    openai_lines.append({
        "messages": [
            {"role": "system",    "content": system_template},
            {"role": "user",      "content": build_user_message(r)},
            {"role": "assistant", "content": json.dumps(r["labels"], ensure_ascii=False)}
        ]
    })

# Write the file
with OPENAI_TRAIN_FILE.open("w", encoding="utf-8") as f:
    for obj in openai_lines:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("\nWrote training JSONL to:", OPENAI_TRAIN_FILE)
print("Example entry:", openai_lines[0])


Input directory created at: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/03_finetuning/all_at_once/openai/gpt-4.1-2025-04-14/runs/2025-11-28_045222_openai_gpt-4.1-2025-04-14_all_at_once_split_v1_seed42/01_inputs
Training file will be: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/03_finetuning/all_at_once/openai/gpt-4.1-2025-04-14/runs/2025-11-28_045222_openai_gpt-4.1-2025-04-14_all_at_once_split_v1_seed42/01_inputs/train.openai.jsonl

Wrote training JSONL to: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/03_finetuning/all_at_once/openai/gpt-4.1-2025-04-14/runs/2025-11-28_045222_openai_gpt-4.1-2025-04-14_all_at_once_split_v1_seed42/01_inputs/train.openai.jsonl
Example entry: {'messages': [{'ro

# 4) Upload Training File to OpenAI
Uploads the JSONL file prepared above.

In [14]:
uploaded = client.files.create(
    file=OPENAI_TRAIN_FILE.open("rb"),
    purpose="fine-tune"
)
train_file_id = uploaded.id
train_file_id

'file-P3VfyK3xsaJkyxWdC5PwD7'

# 5) Create Fine-Tune Job
Start the OpenAI fine-tuning process.

In [15]:
job = client.fine_tuning.jobs.create(
    training_file=train_file_id,
    model=MODEL_NAME,
    suffix=FINETUNE_SUFFIX
)

job_id = job.id
job

FineTuningJob(id='ftjob-7c0qtKRDL2Bd65UajOpBGdPy', created_at=1764305685, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs='auto'), model='gpt-4.1-2025-04-14', object='fine_tuning.job', organization_id='org-dOTRamgeZTttZwK0NvpdUFA2', result_files=[], seed=1565617053, status='validating_files', trained_tokens=None, training_file='file-P3VfyK3xsaJkyxWdC5PwD7', validation_file=None, estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs='auto'))), user_provided_suffix='all_at_once_split_v1_seed42', usage_metrics=None, shared_with_openai=False, eval_id=None)

# 6) Poll Until Training Completes
Wait for OpenAI to finish fine-tuning.

In [ ]:
def wait_for_job(job_id, poll_s=20):
    while True:
        j = client.fine_tuning.jobs.retrieve(job_id)
        print(datetime.now().isoformat(timespec="seconds"), "status =", j.status)
        if j.status in ("succeeded", "failed", "cancelled"):
            return j
        time.sleep(poll_s)

final_job = wait_for_job(job_id)
final_job

2025-11-28T12:55:56 status = validating_files
2025-11-28T12:56:17 status = validating_files
2025-11-28T12:56:37 status = validating_files
2025-11-28T12:56:58 status = validating_files
2025-11-28T12:57:18 status = running
2025-11-28T12:57:39 status = running
2025-11-28T12:57:59 status = running
2025-11-28T12:58:20 status = running
2025-11-28T12:58:40 status = running
2025-11-28T12:59:01 status = running
2025-11-28T12:59:21 status = running
2025-11-28T12:59:42 status = running
2025-11-28T13:00:02 status = running
2025-11-28T13:00:22 status = running
2025-11-28T13:00:43 status = running
2025-11-28T13:01:03 status = running


# 7) Save Run Artifacts
Store all training metadata to a timestamped folder.

In [11]:
import shutil

# ------------------------------------------------------------------
# Create outputs and snapshots directories
# ------------------------------------------------------------------
OUTPUTS_DIR = RUN_DIR / "02_outputs"
SNAPSHOTS_DIR = RUN_DIR / "03_snapshots"

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
SNAPSHOTS_DIR.mkdir(parents=True, exist_ok=True)

print("Outputs dir:   ", OUTPUTS_DIR.resolve())
print("Snapshots dir: ", SNAPSHOTS_DIR.resolve())

# ------------------------------------------------------------------
# Save fine-tune results into outputs/
# ------------------------------------------------------------------
(OUTPUTS_DIR / "model_id.txt").write_text(
    final_job.fine_tuned_model or "",
    encoding="utf-8"
)

(OUTPUTS_DIR / "job.json").write_text(
    json.dumps(final_job.model_dump(), ensure_ascii=False, indent=2),
    encoding="utf-8"
)

# ------------------------------------------------------------------
# Snapshot the notebook itself into 03_snapshots
# ------------------------------------------------------------------
NOTEBOOK_PATH = Path("run_finetune.ipynb")

if NOTEBOOK_PATH.exists():
    # Copy notebook into snapshot folder
    shutil.copy2(NOTEBOOK_PATH, SNAPSHOTS_DIR / "run_finetune_snapshot.ipynb")
    print("Notebook snapshot saved to:", (SNAPSHOTS_DIR / "run_finetune_snapshot.ipynb").resolve())
else:
    print("WARNING: run_finetune.ipynb not found in current directory. Cannot snapshot notebook.")

print("\nSaved artifacts into:", OUTPUTS_DIR.resolve())
print("Full run folder     :", RUN_DIR.resolve())


Outputs dir:    /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/03_finetuning/all_at_once/openai/gpt-4.1-2025-04-14/runs/2025-11-24_105934_openai_gpt-4.1-2025-04-14_all_at_once_split_v1_seed42/02_outputs
Snapshots dir:  /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/03_finetuning/all_at_once/openai/gpt-4.1-2025-04-14/runs/2025-11-24_105934_openai_gpt-4.1-2025-04-14_all_at_once_split_v1_seed42/03_snapshots
Notebook snapshot saved to: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/03_finetuning/all_at_once/openai/gpt-4.1-2025-04-14/runs/2025-11-24_105934_openai_gpt-4.1-2025-04-14_all_at_once_split_v1_seed42/03_snapshots/run_finetune_snapshot.ipynb

Saved artifacts into: /Users/simonhinterreiter/Libra